# MorphoFusion-MLDL

Notebook final del sistema **MorphoFusion-MLDL** para clasificar el nivel de rendimiento deportivo en atletas universitarios mediante variables morfologicas, modelos de Machine Learning, modelos de Deep Learning e interpretabilidad.

> Nota de privacidad: la base de datos original no se incluye en este repositorio porque contiene informacion sensible de participantes. El notebook esta preparado para ejecutarse en Google Colab cargando el archivo Excel local.

## 1. Instalacion e importacion de librerias

Se importan las librerias necesarias para procesamiento de datos, modelado, evaluacion, visualizacion e interpretabilidad.

In [ ]:
!pip install -q xgboost shap openpyxl seaborn

In [ ]:
import os
import re
import shutil
import time
import warnings
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
    matthews_corrcoef,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TEST_SIZE = 0.20
K_FINAL = 91

os.makedirs("resultados_morphofusion/figuras", exist_ok=True)
os.makedirs("resultados_morphofusion/tablas", exist_ok=True)

## 2. Carga de datos

El archivo Excel se carga desde el equipo local. Para proteger la privacidad de los participantes, el notebook no muestra nombres, fechas de nacimiento ni otros identificadores personales.

In [ ]:
uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]

excel = pd.ExcelFile(nombre_archivo)
print("Archivo cargado:", nombre_archivo)
print("Hojas encontradas:", excel.sheet_names)

df_original = pd.read_excel(nombre_archivo, sheet_name=0)

print("Registros cargados:", df_original.shape[0])
print("Columnas cargadas:", df_original.shape[1])
print("Vista previa omitida para proteger datos personales de participantes.")

## 3. Limpieza y preprocesamiento de datos

Se normalizan nombres de columnas, se recodifica la variable objetivo, se excluyen identificadores y variables textuales, y se conserva una matriz numerica para el analisis predictivo.

In [ ]:
def limpiar_nombre_columna(nombre):
    texto = str(nombre).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    texto = texto.replace("%", "porcentaje")
    texto = texto.replace("∑", "suma")
    texto = texto.replace("³", "3")
    texto = re.sub(r"[^a-zA-Z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto


def hacer_nombres_unicos(nombres):
    conteo = {}
    salida = []

    for nombre in nombres:
        if nombre not in conteo:
            conteo[nombre] = 0
            salida.append(nombre)
        else:
            conteo[nombre] += 1
            salida.append(f"{nombre}_{conteo[nombre]}")

    return salida


def normalizar_texto(valor):
    texto = str(valor).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    texto = re.sub(r"[^a-zA-Z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto


df = df_original.copy()
df.columns = hacer_nombres_unicos([limpiar_nombre_columna(col) for col in df.columns])

print("Columnas normalizadas:", len(df.columns))

In [ ]:
mapa_rendimiento = {
    "principiante": 0,
    "pricipiante": 0,
    "intermedio": 0,
    "avanzado": 1,
    "alto_rendimiento": 1,
    "rendimiento": 1,
    "experimentado": 1,
    "titular": 1,
}

if "nivel" not in df.columns:
    raise ValueError("No se encontro la columna 'nivel'. Revise el nombre de la variable objetivo.")

df["nivel_limpio"] = df["nivel"].apply(normalizar_texto)
df["rendimiento_binario"] = df["nivel_limpio"].map(mapa_rendimiento)

niveles_no_mapeados = df.loc[df["rendimiento_binario"].isna(), "nivel"].unique()
if len(niveles_no_mapeados) > 0:
    raise ValueError(f"Existen niveles no mapeados: {niveles_no_mapeados}")

df["rendimiento_binario"] = df["rendimiento_binario"].astype(int)
df["clase_rendimiento"] = df["rendimiento_binario"].map({
    0: "En desarrollo",
    1: "Alto rendimiento",
})

print("Distribucion de la variable objetivo:")
display(df["clase_rendimiento"].value_counts().rename_axis("Clase").reset_index(name="Frecuencia"))

In [ ]:
columnas_excluir = [
    "n", "nombre", "nombre_y_apellidos", "apellidos",
    "universidad", "deporte", "deportes_aplicar", "deporte_adecuado",
    "fecha_eval", "fecha_nac",
    "nivel", "nivel_limpio", "rendimiento_binario", "clase_rendimiento",
    "edad_original", "edad_recalculada", "flag_edad_revisar",
]

patrones_excluir = [
    "evaluacion", "resultado", "resultados", "interpretacion",
    "clasificacion", "categoria", "tipo_de_programa",
]

columnas_textuales = [
    col for col in df.columns
    if any(patron in col for patron in patrones_excluir)
]

columnas_excluir_final = list(set(columnas_excluir + columnas_textuales))

X_raw = df.drop(columns=columnas_excluir_final, errors="ignore")
y = df["rendimiento_binario"].copy()

X = X_raw.apply(lambda col: pd.to_numeric(
    col.astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace("%", "", regex=False)
    .str.strip(),
    errors="coerce",
))

columnas_vacias = X.columns[X.isna().all()].tolist()
X = X.drop(columns=columnas_vacias)

columnas_constantes = [col for col in X.columns if X[col].nunique(dropna=True) <= 1]
X = X.drop(columns=columnas_constantes)

resumen_limpieza = pd.DataFrame({
    "Tecnica aplicada": [
        "Estandarizacion de nombres de columnas",
        "Recodificacion de la variable objetivo",
        "Eliminacion de identificadores y variables textuales",
        "Conversion de variables a formato numerico",
        "Eliminacion de columnas vacias",
        "Eliminacion de columnas constantes",
        "Base final para analisis",
    ],
    "Evidencia": [
        f"{df_original.shape[1]} columnas normalizadas",
        "Nivel recodificado en 0 = En desarrollo y 1 = Alto rendimiento",
        f"{len(columnas_excluir_final)} columnas excluidas",
        f"{X_raw.shape[1]} variables candidatas procesadas",
        f"{len(columnas_vacias)} columnas eliminadas",
        f"{len(columnas_constantes)} columnas eliminadas",
        f"{X.shape[0]} registros y {X.shape[1]} variables predictoras candidatas",
    ],
})

display(resumen_limpieza)
resumen_limpieza.to_excel("resultados_morphofusion/tablas/resumen_limpieza_datos.xlsx", index=False)

## 4. Agrupacion de variables morfologicas

Las variables predictoras se agrupan segun su dimension morfologica para describir la composicion del conjunto de datos usado en el modelado.

In [ ]:
def limpiar_texto_dimension(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    texto = re.sub(r"[^a-zA-Z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto


def asignar_dimension(variable):
    v = limpiar_texto_dimension(variable)

    if any(p in v for p in ["edad", "maduracion", "pico_veloc", "crecimiento", "altura_adulta", "talla_diana", "phv"]):
        return "Maduracion biologica"
    if any(p in v for p in ["peso", "talla", "talla_sentado", "envergadura", "estatura_padre", "estatura_madre"]):
        return "Antropometria basica"
    if any(p in v for p in ["biacro", "bilio", "humer", "femor", "tibial", "trocanteira", "radial", "acromial", "acromio", "long_pie", "long_mano", "t_transv", "t_ap"]):
        return "Diametros y longitudes oseas"
    if any(p in v for p in ["cbz", "brrel", "brflex", "antebr", "torax", "cintura", "cademax", "musmax", "pantmax", "arm_g", "calf_g"]):
        return "Perimetros corporales"
    if any(p in v for p in ["trc", "ssc", "ssp", "abd", "mmed", "pant", "suma6p", "suma3p"]):
        return "Pliegues cutaneos"
    if any(p in v for p in ["area_total", "musc_brazo", "musc_muslo", "musc_pierna", "adip_brazo", "adip_muslo", "adip_pierna", "total_antebr", "grasa_del_brazo"]):
        return "Areas musculares y adiposas"
    if any(p in v for p in ["masa", "grasa", "adiposa", "muscular", "osea", "piel", "lipidica", "lipido", "mca", "endo", "meso", "ecto", "peso_graso"]):
        return "Composicion corporal"
    if any(p in v for p in ["indice", "icormico", "irmi", "rohrer", "pignet", "aks", "hwr", "imo", "iam", "iecto", "ip_m", "x_coord", "y_coord"]):
        return "Proporcionalidad e indices morfologicos"

    return "Revisar manualmente"


tabla_variables = pd.DataFrame({"Variable": X.columns})
tabla_variables["Dimension morfologica"] = tabla_variables["Variable"].apply(asignar_dimension)
tabla_variables["Usar en el modelo"] = np.where(
    tabla_variables["Dimension morfologica"] == "Revisar manualmente",
    "Revisar",
    "Si",
)

resumen_dimensiones = (
    tabla_variables[tabla_variables["Usar en el modelo"] == "Si"]
    ["Dimension morfologica"]
    .value_counts()
    .reset_index()
)
resumen_dimensiones.columns = ["Dimension morfologica", "Numero de variables"]

display(resumen_dimensiones)
tabla_variables.to_excel("resultados_morphofusion/tablas/variables_por_dimension.xlsx", index=False)

In [ ]:
plt.figure(figsize=(9, 5), dpi=150)
sns.barplot(
    data=resumen_dimensiones,
    x="Numero de variables",
    y="Dimension morfologica",
    color="#4B6F8C",
)
plt.title("Distribution of Predictor Variables by Morphological Dimension", fontsize=13, weight="bold")
plt.xlabel("Number of variables")
plt.ylabel("")

for index, value in enumerate(resumen_dimensiones["Numero de variables"]):
    plt.text(value + 0.5, index, str(value), va="center", fontsize=10)

plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/variables_por_dimension.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Analisis de correlacion de Pearson

Se calcula una matriz de correlacion para explorar asociaciones lineales entre variables morfologicas y detectar posibles redundancias.

In [ ]:
imputador_correlacion = SimpleImputer(strategy="median")
X_imputado = pd.DataFrame(
    imputador_correlacion.fit_transform(X),
    columns=X.columns,
    index=X.index,
)

df_corr = X_imputado.copy()
df_corr["rendimiento_binario"] = y.values

correlacion_objetivo = (
    df_corr.corr(numeric_only=True)["rendimiento_binario"]
    .drop("rendimiento_binario")
    .abs()
    .sort_values(ascending=False)
)

top_n = min(10, len(correlacion_objetivo))
variables_corr = correlacion_objetivo.head(top_n).index.tolist()
matriz_correlacion = df_corr[variables_corr].corr().round(2)

tabla_correlacion_objetivo = correlacion_objetivo.reset_index()
tabla_correlacion_objetivo.columns = ["Variable", "Correlacion_absoluta_con_rendimiento"]
display(tabla_correlacion_objetivo.head(20))

tabla_correlacion_objetivo.to_excel(
    "resultados_morphofusion/tablas/correlacion_con_rendimiento.xlsx",
    index=False,
)

In [ ]:
def etiqueta_corta(texto, max_len=24):
    texto = str(texto)
    return texto if len(texto) <= max_len else texto[:max_len - 3] + "..."

etiquetas = [etiqueta_corta(v) for v in variables_corr]

fig, ax = plt.subplots(figsize=(10, 8), dpi=200)
im = ax.imshow(matriz_correlacion.values, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(np.arange(len(variables_corr)))
ax.set_yticks(np.arange(len(variables_corr)))
ax.set_xticklabels(etiquetas, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(etiquetas, fontsize=8)

ax.set_xticks(np.arange(-0.5, len(variables_corr), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(variables_corr), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.2)
ax.tick_params(which="minor", bottom=False, left=False)

for i in range(len(variables_corr)):
    for j in range(len(variables_corr)):
        valor = matriz_correlacion.iloc[i, j]
        color_texto = "white" if abs(valor) >= 0.55 else "black"
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", color=color_texto, fontsize=9, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Pearson correlation coefficient", fontsize=9)

ax.set_title("Pearson Correlation Map of Selected Morphological Variables", fontsize=13, weight="bold", pad=14)
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/mapa_correlacion_pearson.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Division entrenamiento-prueba y seleccion de caracteristicas

Se usa una particion estratificada 80/20. La seleccion de caracteristicas se realiza con SelectKBest y ANOVA F-value, conservando hasta 91 variables.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

variables_numericas = X.columns.tolist()
k_final = min(K_FINAL, len(variables_numericas))

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Variables candidatas:", len(variables_numericas))
print("Variables seleccionadas:", k_final)

In [ ]:
def crear_preprocesamiento_y_selector():
    preprocesamiento = ColumnTransformer(
        transformers=[
            ("numericas", Pipeline(steps=[
                ("imputacion_mediana", SimpleImputer(strategy="median")),
                ("escalamiento", StandardScaler()),
            ]), variables_numericas),
        ],
        remainder="drop",
    )

    selector = SelectKBest(score_func=f_classif, k=k_final)

    return Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("seleccion_caracteristicas", selector),
    ])

## 7. Entrenamiento de modelos de Machine Learning

Se entrenan cuatro modelos supervisados: Regresion Logistica, SVM, Random Forest y XGBoost. Para evidenciar el entrenamiento, el notebook registra la ejecucion de `fit()`, el tiempo de entrenamiento de cada modelo, el numero de registros usados y las metricas obtenidas en el conjunto de prueba.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

modelos_ml = {
    "Regresion Logistica": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        verbosity=0,
    ),
}

In [ ]:
resultados_ml_final = []
matrices_ml_final = {}
predicciones_ml_final = {}
probabilidades_ml_final = {}
pipelines_ml_final = {}
tiempos_entrenamiento_ml = []

for nombre_modelo, modelo in modelos_ml.items():
    print(f"Entrenando modelo ML: {nombre_modelo}")
    inicio = time.time()

    pipeline = Pipeline(steps=[
        ("preprocesamiento_y_seleccion", crear_preprocesamiento_y_selector()),
        ("modelo", modelo),
    ])

    pipeline.fit(X_train, y_train)
    duracion = time.time() - inicio

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    matriz = confusion_matrix(y_test, y_pred)

    resultados_ml_final.append({
        "Modelo": nombre_modelo,
        "k_variables": k_final,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_test, y_prob),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })

    tiempos_entrenamiento_ml.append({
        "Modelo": nombre_modelo,
        "Entrenamiento ejecutado": "Si",
        "Registros entrenamiento": X_train.shape[0],
        "Registros prueba": X_test.shape[0],
        "Variables seleccionadas": k_final,
        "Tiempo entrenamiento segundos": round(duracion, 3),
    })

    matrices_ml_final[nombre_modelo] = matriz
    predicciones_ml_final[nombre_modelo] = y_pred
    probabilidades_ml_final[nombre_modelo] = y_prob
    pipelines_ml_final[nombre_modelo] = pipeline

    print(f"Modelo entrenado: {nombre_modelo} | tiempo: {duracion:.2f} s")

resultados_ml_final = pd.DataFrame(resultados_ml_final).sort_values(
    ["F1-score", "Balanced accuracy", "AUC-ROC"],
    ascending=False,
)

tabla_entrenamiento_ml = pd.DataFrame(tiempos_entrenamiento_ml)

display(tabla_entrenamiento_ml)
display(resultados_ml_final.round(3))

tabla_entrenamiento_ml.to_excel("resultados_morphofusion/tablas/evidencia_entrenamiento_ml.xlsx", index=False)
resultados_ml_final.to_excel("resultados_morphofusion/tablas/resultados_machine_learning.xlsx", index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 8), dpi=180)
axes = axes.flatten()

for ax, nombre_modelo in zip(axes, modelos_ml.keys()):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=matrices_ml_final[nombre_modelo],
        display_labels=["Developing", "High performance"],
    )
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(nombre_modelo)

plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/matrices_confusion_ml.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6), dpi=150)

for nombre_modelo, y_prob in probabilidades_ml_final.items():
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2.3, label=f"{nombre_modelo} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5, label="Chance")
plt.title("ROC Curve of Machine Learning Models", fontsize=13, weight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/roc_machine_learning.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
metricas_ml_grafico = resultados_ml_final.melt(
    id_vars="Modelo",
    value_vars=["Accuracy", "Balanced accuracy", "F1-score", "AUC-ROC"],
    var_name="Metrica",
    value_name="Valor",
)

plt.figure(figsize=(10, 5.5), dpi=150)
sns.barplot(data=metricas_ml_grafico, x="Modelo", y="Valor", hue="Metrica")
plt.title("Comparison of Machine Learning Model Performance", fontsize=13, weight="bold")
plt.ylabel("Metric value")
plt.xlabel("")
plt.ylim(0, 1)
plt.legend(title="")
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/comparacion_metricas_ml.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Validacion cruzada de modelos de Machine Learning

Se aplica validacion cruzada estratificada para evaluar la estabilidad del rendimiento de los modelos.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

resultados_cv = []
for nombre_modelo, modelo in modelos_ml.items():
    pipeline = Pipeline(steps=[
        ("preprocesamiento_y_seleccion", crear_preprocesamiento_y_selector()),
        ("modelo", modelo),
    ])

    scores = cross_validate(pipeline, X, y, cv=cv5, scoring=scoring)

    resultados_cv.append({
        "Modelo": nombre_modelo,
        "Accuracy promedio": scores["test_accuracy"].mean(),
        "Balanced accuracy promedio": scores["test_balanced_accuracy"].mean(),
        "F1 promedio": scores["test_f1"].mean(),
        "AUC-ROC promedio": scores["test_roc_auc"].mean(),
    })

tabla_cv = pd.DataFrame(resultados_cv).sort_values("F1 promedio", ascending=False)
display(tabla_cv.round(3))
tabla_cv.to_excel("resultados_morphofusion/tablas/validacion_cruzada_ml.xlsx", index=False)

## 9. Entrenamiento de modelos de Deep Learning

Se entrenan tres arquitecturas: MLP, CNN1D y un modelo con mecanismo de atencion. Para evidenciar el entrenamiento, el notebook conserva los historiales de Keras (`history`), registra epocas ejecutadas, tiempos de entrenamiento y genera curvas de perdida y accuracy para entrenamiento y validacion.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Conv1D,
    GlobalAveragePooling1D,
    MultiHeadAttention,
    Add,
    LayerNormalization,
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(RANDOM_STATE)

imputador_dl = SimpleImputer(strategy="median")
X_train_imputado = imputador_dl.fit_transform(X_train)
X_test_imputado = imputador_dl.transform(X_test)

escalador_dl = StandardScaler()
X_train_dl = escalador_dl.fit_transform(X_train_imputado)
X_test_dl = escalador_dl.transform(X_test_imputado)

X_train_cnn = X_train_dl.reshape(X_train_dl.shape[0], X_train_dl.shape[1], 1)
X_test_cnn = X_test_dl.reshape(X_test_dl.shape[0], X_test_dl.shape[1], 1)

pesos_clase = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
class_weight = {int(clase): float(peso) for clase, peso in zip(np.unique(y_train), pesos_clase)}

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
)

print("X_train_dl:", X_train_dl.shape)
print("X_test_dl:", X_test_dl.shape)
print("Class weights:", class_weight)

In [ ]:
def evaluar_modelo_dl(nombre_modelo, modelo, X_eval, y_eval):
    y_prob = modelo.predict(X_eval, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    metricas = {
        "Modelo": nombre_modelo,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1-score": f1_score(y_eval, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_eval, y_prob),
        "MCC": matthews_corrcoef(y_eval, y_pred),
    }

    matriz = confusion_matrix(y_eval, y_pred)
    return metricas, matriz, y_pred, y_prob

In [ ]:
print("Entrenando modelo DL: MLP")
inicio_mlp = time.time()

modelo_mlp = Sequential([
    Input(shape=(X_train_dl.shape[1],)),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.30),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid"),
])

modelo_mlp.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

historial_mlp = modelo_mlp.fit(
    X_train_dl,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=8,
    callbacks=[early_stop],
    class_weight=class_weight,
    verbose=0,
)

duracion_mlp = time.time() - inicio_mlp
print(f"MLP entrenado en {duracion_mlp:.2f} s | epocas ejecutadas: {len(historial_mlp.history['loss'])}")

metricas_mlp, matriz_mlp, y_pred_mlp, y_prob_mlp = evaluar_modelo_dl(
    "MLP", modelo_mlp, X_test_dl, y_test
)

In [ ]:
print("Entrenando modelo DL: CNN1D")
inicio_cnn1d = time.time()

modelo_cnn1d = Sequential([
    Input(shape=(X_train_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=3, activation="relu", padding="same"),
    BatchNormalization(),
    Dropout(0.30),
    Conv1D(filters=16, kernel_size=3, activation="relu", padding="same"),
    GlobalAveragePooling1D(),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid"),
])

modelo_cnn1d.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

historial_cnn1d = modelo_cnn1d.fit(
    X_train_cnn,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=8,
    callbacks=[early_stop],
    class_weight=class_weight,
    verbose=0,
)

duracion_cnn1d = time.time() - inicio_cnn1d
print(f"CNN1D entrenado en {duracion_cnn1d:.2f} s | epocas ejecutadas: {len(historial_cnn1d.history['loss'])}")

metricas_cnn1d, matriz_cnn1d, y_pred_cnn1d, y_prob_cnn1d = evaluar_modelo_dl(
    "CNN1D", modelo_cnn1d, X_test_cnn, y_test
)

In [ ]:
print("Entrenando modelo DL: Attention")
inicio_attention = time.time()

entrada = Input(shape=(X_train_cnn.shape[1], 1))
x = Conv1D(filters=32, kernel_size=3, activation="relu", padding="same")(entrada)
x = BatchNormalization()(x)
attn = MultiHeadAttention(num_heads=2, key_dim=16)(x, x)
x = Add()([x, attn])
x = LayerNormalization()(x)
x = GlobalAveragePooling1D()(x)
x = Dense(32, activation="relu")(x)
x = Dropout(0.25)(x)
salida = Dense(1, activation="sigmoid")(x)

modelo_attention = Model(inputs=entrada, outputs=salida)
modelo_attention.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

historial_attention = modelo_attention.fit(
    X_train_cnn,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=8,
    callbacks=[early_stop],
    class_weight=class_weight,
    verbose=0,
)

duracion_attention = time.time() - inicio_attention
print(f"Attention entrenado en {duracion_attention:.2f} s | epocas ejecutadas: {len(historial_attention.history['loss'])}")

metricas_attention, matriz_attention, y_pred_attention, y_prob_attention = evaluar_modelo_dl(
    "Attention", modelo_attention, X_test_cnn, y_test
)

In [ ]:
resultados_dl = pd.DataFrame([
    metricas_mlp,
    metricas_attention,
    metricas_cnn1d,
]).sort_values(["F1-score", "Balanced accuracy", "AUC-ROC"], ascending=False)

matrices_dl = {
    "MLP": matriz_mlp,
    "Attention": matriz_attention,
    "CNN1D": matriz_cnn1d,
}

probabilidades_dl = {
    "MLP": y_prob_mlp,
    "Attention": y_prob_attention,
    "CNN1D": y_prob_cnn1d,
}

historiales_dl = {
    "MLP": historial_mlp,
    "Attention": historial_attention,
    "CNN1D": historial_cnn1d,
}

tabla_entrenamiento_dl = pd.DataFrame([
    {
        "Modelo": "MLP",
        "Entrenamiento ejecutado": "Si",
        "Epocas ejecutadas": len(historial_mlp.history["loss"]),
        "Batch size": 8,
        "Tiempo entrenamiento segundos": round(duracion_mlp, 3),
    },
    {
        "Modelo": "Attention",
        "Entrenamiento ejecutado": "Si",
        "Epocas ejecutadas": len(historial_attention.history["loss"]),
        "Batch size": 8,
        "Tiempo entrenamiento segundos": round(duracion_attention, 3),
    },
    {
        "Modelo": "CNN1D",
        "Entrenamiento ejecutado": "Si",
        "Epocas ejecutadas": len(historial_cnn1d.history["loss"]),
        "Batch size": 8,
        "Tiempo entrenamiento segundos": round(duracion_cnn1d, 3),
    },
])

display(tabla_entrenamiento_dl)
display(resultados_dl.round(3))

tabla_entrenamiento_dl.to_excel("resultados_morphofusion/tablas/evidencia_entrenamiento_dl.xlsx", index=False)
resultados_dl.to_excel("resultados_morphofusion/tablas/resultados_deep_learning.xlsx", index=False)

### Evidencia grafica del entrenamiento Deep Learning

Las curvas de perdida y accuracy muestran la evolucion del entrenamiento y validacion de MLP, Attention y CNN1D. Estas figuras sirven como evidencia adicional del proceso de entrenamiento de las arquitecturas neuronales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), dpi=150)

for nombre_modelo, historial in historiales_dl.items():
    axes[0].plot(historial.history["loss"], label=f"{nombre_modelo} - train")
    axes[0].plot(historial.history["val_loss"], linestyle="--", label=f"{nombre_modelo} - val")

    axes[1].plot(historial.history["accuracy"], label=f"{nombre_modelo} - train")
    axes[1].plot(historial.history["val_accuracy"], linestyle="--", label=f"{nombre_modelo} - val")

axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_title("Training and Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle("Training Curves of Deep Learning Models", fontsize=13, weight="bold")
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/curvas_entrenamiento_deep_learning.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=180)

for ax, nombre_modelo in zip(axes, matrices_dl.keys()):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=matrices_dl[nombre_modelo],
        display_labels=["Developing", "High performance"],
    )
    disp.plot(ax=ax, cmap="Greens", values_format="d", colorbar=False)
    ax.set_title(nombre_modelo)

plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/matrices_confusion_dl.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6), dpi=150)

for nombre_modelo, y_prob in probabilidades_dl.items():
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2.3, label=f"{nombre_modelo} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5, label="Chance")
plt.title("ROC Curve of Deep Learning Models", fontsize=13, weight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/roc_deep_learning.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Comparacion global de modelos

Se comparan conjuntamente los modelos de Machine Learning y Deep Learning usando las mismas metricas de evaluacion.

In [ ]:
tabla_ml_final = resultados_ml_final.drop(columns=["k_variables"], errors="ignore").copy()
tabla_ml_final["Tipo de modelo"] = "Machine Learning"

tabla_dl_final = resultados_dl.copy()
tabla_dl_final["Tipo de modelo"] = "Deep Learning"

tabla_comparacion_final = pd.concat([tabla_ml_final, tabla_dl_final], ignore_index=True)
tabla_comparacion_final = tabla_comparacion_final.sort_values(
    ["F1-score", "Balanced accuracy", "AUC-ROC"],
    ascending=False,
)

columnas_metricas = ["Accuracy", "Balanced accuracy", "Precision", "Recall", "F1-score", "AUC-ROC", "MCC"]
tabla_comparacion_final[columnas_metricas] = tabla_comparacion_final[columnas_metricas].round(3)

display(tabla_comparacion_final)
tabla_comparacion_final.to_excel("resultados_morphofusion/tablas/comparacion_global_modelos.xlsx", index=False)

In [ ]:
metricas_globales = tabla_comparacion_final.melt(
    id_vars=["Modelo", "Tipo de modelo"],
    value_vars=["F1-score", "Balanced accuracy"],
    var_name="Metrica",
    value_name="Valor",
)

plt.figure(figsize=(10, 5.5), dpi=150)
sns.barplot(data=metricas_globales, x="Modelo", y="Valor", hue="Metrica")
plt.title("Global Comparison of ML and DL Models", fontsize=13, weight="bold")
plt.ylabel("Metric value")
plt.xlabel("")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.legend(title="")
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/comparacion_global_modelos.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
probabilidades_todos_modelos = {}
probabilidades_todos_modelos.update(probabilidades_ml_final)
probabilidades_todos_modelos.update(probabilidades_dl)

plt.figure(figsize=(9, 7), dpi=150)

for nombre_modelo, y_prob in probabilidades_todos_modelos.items():
    fpr, tpr, thresholds = roc_curve(y_test, np.ravel(y_prob))
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2.2, label=f"{nombre_modelo} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5, label="Chance")
plt.title("General ROC Curve of the Evaluated Models", fontsize=13, weight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/roc_general_modelos.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Interpretabilidad del modelo seleccionado

Se selecciona Regresion Logistica como modelo interpretable y se analizan los coeficientes y valores SHAP para identificar variables influyentes.

In [ ]:
modelo_final_nombre = "Regresion Logistica"
pipeline_logistica = pipelines_ml_final[modelo_final_nombre]

prep_selector = pipeline_logistica.named_steps["preprocesamiento_y_seleccion"]
modelo_logistica = pipeline_logistica.named_steps["modelo"]

X_train_proc = prep_selector.transform(X_train)
X_test_proc = prep_selector.transform(X_test)

selector = prep_selector.named_steps["seleccion_caracteristicas"]
mascara_variables = selector.get_support()
nombres_variables_proc = np.array(variables_numericas)[mascara_variables]

coeficientes = modelo_logistica.coef_[0]

importancia_logistica = pd.DataFrame({
    "Variable": nombres_variables_proc,
    "Coeficiente": coeficientes,
    "Importancia absoluta": np.abs(coeficientes),
}).sort_values("Importancia absoluta", ascending=False)

display(importancia_logistica.head(20))
importancia_logistica.to_excel("resultados_morphofusion/tablas/importancia_regresion_logistica.xlsx", index=False)

In [ ]:
top_coeficientes = importancia_logistica.head(15).sort_values("Coeficiente")

plt.figure(figsize=(8, 7), dpi=150)
colores = np.where(top_coeficientes["Coeficiente"] >= 0, "#4B6F8C", "#C75C5C")
plt.barh(top_coeficientes["Variable"], top_coeficientes["Coeficiente"], color=colores)
plt.axvline(0, color="gray", linewidth=1)
plt.title("Direction of Influence of Morphological Variables", fontsize=13, weight="bold")
plt.xlabel("Logistic regression coefficient")
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/direccion_influencia_logistica.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import shap

explainer = shap.LinearExplainer(modelo_logistica, X_train_proc)
shap_values = explainer.shap_values(X_test_proc)

plt.figure(figsize=(9, 6), dpi=150)
shap.summary_plot(
    shap_values,
    X_test_proc,
    feature_names=nombres_variables_proc,
    show=False,
    max_display=15,
)
plt.tight_layout()
plt.savefig("resultados_morphofusion/figuras/shap_summary_plot.png", dpi=300, bbox_inches="tight")
plt.show()

importancia_shap = pd.DataFrame({
    "Variable": nombres_variables_proc,
    "SHAP absoluto medio": np.abs(shap_values).mean(axis=0),
}).sort_values("SHAP absoluto medio", ascending=False)

display(importancia_shap.head(20))
importancia_shap.to_excel("resultados_morphofusion/tablas/importancia_shap.xlsx", index=False)

## 12. Exportacion de resultados

Se comprimen las tablas y figuras generadas para facilitar su descarga desde Colab.

In [ ]:
shutil.make_archive("resultados_morphofusion", "zip", "resultados_morphofusion")
files.download("resultados_morphofusion.zip")